In [1]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


2.5.1+cu121
True
NVIDIA GeForce GTX 1050


In [2]:
!pip install -q faiss-cpu
!pip install -q sentencepiece
!pip install -q sentence-transformers
!pip install -q scikit-learn



In [3]:
import faiss
import sklearn
import sentencepiece as spm
from sentence_transformers import SentenceTransformer

print("FAISS:", faiss.__version__)
print("sentencepiece loaded")
print("sklearn:", sklearn.__version__)
print("SentenceTransformer loaded")



FAISS: 1.13.2
sentencepiece loaded
sklearn: 1.7.2
SentenceTransformer loaded


In [4]:
import os

PROJECT_ROOT = "D:\Downloads_D\Projects\SLM\Banking_SLM_Project"
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
MEMORY_DIR = os.path.join(PROJECT_ROOT, "memory_bank")
LOG_DIR = os.path.join(PROJECT_ROOT, "logs")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(MEMORY_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print("Project directories created:")
print(PROJECT_ROOT)

Project directories created:
D:\Downloads_D\Projects\SLM\Banking_SLM_Project


In [5]:
# =========================
# Standard Library
# =========================
import os
import re
import json
import time
import math
import random
import hashlib
from collections import deque, defaultdict
from datetime import datetime

# =========================
# Numerics
# =========================
import numpy as np
import sklearn
from sklearn.decomposition import PCA

# =========================
# Torch
# =========================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# =========================
# FAISS
# =========================
import faiss

# =========================
# Tokeniser
# =========================
import sentencepiece as spm

# =========================
# Embeddings
# =========================
from sentence_transformers import SentenceTransformer

# =========================
# HuggingFace Datasets
# =========================
from datasets import load_dataset

print("All imports successful.")



All imports successful.


## Tokenizer

We use OpenAI’s cl100k_base because:

*   Strong financial vocabulary coverage
*   Efficient BPE
*   Stable production tokenizer

In [6]:
# Tokenizer config (SentencePiece 32k)

TARGET_VOCAB_SIZE = 32000
SPECIAL_TOKEN_STRINGS = ["<|system|>", "<|user|>", "<|assistant|>", "<|end|>"]

TOKENIZER_DIR = os.path.join(PROJECT_ROOT, "tokenizer")
TOKENIZER_PREFIX = os.path.join(TOKENIZER_DIR, "spm_32k")
TOKENIZER_MODEL_PATH = TOKENIZER_PREFIX + ".model"
TOKENIZER_VOCAB_PATH = TOKENIZER_PREFIX + ".vocab"
os.makedirs(TOKENIZER_DIR, exist_ok=True)

sp = None
SPECIAL_TOKENS = {}
SPECIAL_TOKEN_IDS = set()
VOCAB_SIZE = None
BASE_VOCAB_SIZE = None

print("Tokenizer target vocab:", TARGET_VOCAB_SIZE)
print("Tokenizer model path:", TOKENIZER_MODEL_PATH)



Tokenizer target vocab: 32000
Tokenizer model path: D:\Downloads_D\Projects\SLM\Banking_SLM_Project\tokenizer\spm_32k.model


We manually assign IDs ABOVE base vocab boundary.

Why?
*  Prevent collision with base vocabulary
*  Ensure consistent reproducibility
*  Required for weight tying later

We define 4 special tokens:

<|system|>
<|user|>
<|assistant|>
<|end|>

In [7]:
# Load existing tokenizer if present

if os.path.exists(TOKENIZER_MODEL_PATH):
    sp = spm.SentencePieceProcessor(model_file=TOKENIZER_MODEL_PATH)
    SPECIAL_TOKENS = {tok: sp.piece_to_id(tok) for tok in SPECIAL_TOKEN_STRINGS}
    SPECIAL_TOKEN_IDS = set(SPECIAL_TOKENS.values())
    VOCAB_SIZE = sp.get_piece_size()
    BASE_VOCAB_SIZE = VOCAB_SIZE - len(SPECIAL_TOKENS)

    print("Loaded tokenizer from disk.")
    print("Special tokens:", SPECIAL_TOKENS)
    print("Total vocab size:", VOCAB_SIZE)
else:
    print("Tokenizer model not found yet. Run the tokenizer training cell after dataset assembly.")



Tokenizer model not found yet. Run the tokenizer training cell after dataset assembly.


In [8]:
# Validate tokenizer setup (optional)

if sp is not None:
    print("Tokenizer ready.")
    print("UNK id:", sp.unk_id())
    print("Piece size:", sp.get_piece_size())
else:
    print("Tokenizer not loaded yet.")



Tokenizer not loaded yet.


In [9]:
def tokenize(text):
    if sp is None:
        raise RuntimeError("Tokenizer is not loaded. Run the SentencePiece training/tokenization cell first.")
    return sp.encode(text, out_type=int)



In [10]:
def detokenize(token_ids):
    if sp is None:
        raise RuntimeError("Tokenizer is not loaded. Run the SentencePiece training/tokenization cell first.")

    filtered = [tid for tid in token_ids if tid not in SPECIAL_TOKEN_IDS and tid >= 0]
    return sp.decode(filtered)



In [11]:
test_text = (
    "<|system|> You are a banking assistant. "
    "<|user|> What is RBI policy on loans? "
    "<|assistant|> I will check that for you. "
    "<|end|>"
)

if sp is None:
    print("Tokenizer not loaded yet. Run the SentencePiece training/tokenization cell first.")
else:
    tokens = tokenize(test_text)
    reconstructed = detokenize(tokens)

    print("Original:", test_text)
    print("Reconstructed:", reconstructed)
    print("Token IDs sample:", tokens[:20])



Tokenizer not loaded yet. Run the SentencePiece training/tokenization cell first.


## Datasets
1️⃣ banking77

Purpose → Banking intent grounding
Size → ~13k
Structure → short customer banking queries + labels
Role → domain intent awareness

2️⃣ yahma/alpaca-cleaned (first 50k streamed)

Purpose → Instruction-following capability
Size → 50,000
Structure → instruction + optional input + output
Role → teach conversational reasoning structure

3️⃣ zeroshot/twitter-financial-news-sentiment

Purpose → Financial sentiment understanding
Structure → financial news text + sentiment label
Role → teach tone and financial language nuance

🔹 Why This Combination Works

Even though we replaced Phrasebank:
Banking77 → domain grounding
Alpaca → conversational competence
Twitter financial sentiment → financial language realism

For a 12M model, this is actually a stronger mix than the original trio.

Domain knowledge will primarily live in:
👉 Your FAISS partitions
👉 Memory indexing
👉 Post-training SLM embedding re-index

In [12]:
banking_ds = load_dataset("banking77", split="train")

print("Banking77 size:", len(banking_ds))
banking_ds[0]


Banking77 size: 10003


{'text': 'I am still waiting on my card?', 'label': 11}

In [13]:
def format_banking(example):
    user = example["text"].strip()
    label = banking_ds.features["label"].int2str(example["label"])
    assistant = f"Intent: {label}."
    return f"<|user|> {user} <|assistant|> {assistant} <|end|>"

banking_formatted = [format_banking(ex) for ex in banking_ds]

print("Banking formatted examples:", len(banking_formatted))
print(banking_formatted[0])



Banking formatted examples: 10003
<|user|> I am still waiting on my card? <|assistant|> Intent: card_arrival. <|end|>


In [14]:
# intentionally left blank (cleaned)


In [15]:
finance_stream = load_dataset(
    "yahma/alpaca-cleaned",
    split="train",
    streaming=True
)

finance_examples = []
for i, example in enumerate(finance_stream):
    finance_examples.append(example)
    if i >= 49999:
        break

print("Instruction dataset loaded:", len(finance_examples))
print(finance_examples[0])

Instruction dataset loaded: 50000
{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.', 'input': '', 'instruction': 'Give three tips for staying healthy.'}


In [16]:
def format_finance(example):
    user_text = (example["instruction"] or "").strip()
    input_text = (example.get("input") or "").strip()
    assistant = (example.get("output") or "").strip()

    if input_text:
        user_text = f"{user_text} {input_text}".strip()

    return f"<|user|> {user_text} <|assistant|> {assistant} <|end|>"

finance_formatted = [format_finance(ex) for ex in finance_examples]
print("Finance formatted examples:", len(finance_formatted))



Finance formatted examples: 50000


In [17]:
sentiment_ds = load_dataset(
    "zeroshot/twitter-financial-news-sentiment",
    split="train"
)

print("Sentiment dataset size:", len(sentiment_ds))
print(sentiment_ds[0])

Sentiment dataset size: 9543
{'text': '$BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT', 'label': 0}


In [18]:
label_map = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

def format_sentiment(example):
    user_text = example["text"].strip()
    sentiment = label_map[int(example["label"])]
    assistant_text = f"Sentiment: {sentiment}."
    return f"<|user|> {user_text} <|assistant|> {assistant_text} <|end|>"

sentiment_formatted = [format_sentiment(ex) for ex in sentiment_ds]

print("Formatted sentiment examples:", len(sentiment_formatted))
print(sentiment_formatted[0])



Formatted sentiment examples: 9543
<|user|> $BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT <|assistant|> Sentiment: negative. <|end|>


In [19]:
# Build a cleaner, domain-focused corpus (faster + more consistent)

finance_keywords = [
    "bank", "banking", "loan", "credit", "debit", "account", "interest",
    "fraud", "kyc", "rbi", "compliance", "payment", "upi", "card", "mortgage",
    "deposit", "withdraw", "emi", "risk", "insurance", "finance", "financial",
]

def is_finance_related(text):
    t = text.lower()
    return any(k in t for k in finance_keywords)

finance_filtered = [x for x in finance_formatted if is_finance_related(x)]

# Keep a controlled amount of generic instruction data so it does not dominate.
finance_filtered = finance_filtered[:20000]

all_examples = banking_formatted + finance_filtered + sentiment_formatted

print("Banking examples:", len(banking_formatted))
print("Finance filtered examples:", len(finance_filtered))
print("Sentiment examples:", len(sentiment_formatted))
print("Total examples:", len(all_examples))



Banking examples: 10003
Finance filtered examples: 9840
Sentiment examples: 9543
Total examples: 29386


In [20]:
# Train/validation split by example (95/5)

random.seed(42)
random.shuffle(all_examples)

split_idx = int(0.95 * len(all_examples))
train_examples = all_examples[:split_idx]
val_examples = all_examples[split_idx:]

print("Train examples:", len(train_examples))
print("Val examples:", len(val_examples))



Train examples: 27916
Val examples: 1470


This is one of the most important engineering steps.

We will:

*   Tokenise everything
*   Flatten into one large array
*   Save as uint32
*   Create memory-mapped dataset class








In [21]:
# Train/load SentencePiece tokenizer (32k) and tokenize train/val corpora

def ensure_end_token(text):
    t = text.strip()
    return t if t.endswith("<|end|>") else f"{t} <|end|>"

train_examples = [ensure_end_token(x) for x in train_examples]
val_examples = [ensure_end_token(x) for x in val_examples]

tokenizer_train_text_path = os.path.join(TOKENIZER_DIR, "spm_train_text.txt")
with open(tokenizer_train_text_path, "w", encoding="utf-8") as f:
    for example in train_examples:
        f.write(example.replace("\n", " ").strip() + "\n")

if not os.path.exists(TOKENIZER_MODEL_PATH):
    print("Training SentencePiece tokenizer...")
    spm.SentencePieceTrainer.train(
        input=tokenizer_train_text_path,
        model_prefix=TOKENIZER_PREFIX,
        vocab_size=TARGET_VOCAB_SIZE,
        model_type="unigram",
        character_coverage=1.0,
        unk_id=0,
        bos_id=-1,
        eos_id=-1,
        pad_id=-1,
        user_defined_symbols=",".join(SPECIAL_TOKEN_STRINGS),
    )

sp = spm.SentencePieceProcessor(model_file=TOKENIZER_MODEL_PATH)
SPECIAL_TOKENS = {tok: sp.piece_to_id(tok) for tok in SPECIAL_TOKEN_STRINGS}
SPECIAL_TOKEN_IDS = set(SPECIAL_TOKENS.values())
VOCAB_SIZE = sp.get_piece_size()
BASE_VOCAB_SIZE = VOCAB_SIZE - len(SPECIAL_TOKENS)

print("Tokenizer loaded.")
print("Special tokens:", SPECIAL_TOKENS)
print("Total vocab size:", VOCAB_SIZE)

def encode_corpus(examples):
    ids = []
    for ex in examples:
        ids.extend(tokenize(ex))
    return np.array(ids, dtype=np.uint32)

train_token_array = encode_corpus(train_examples)
val_token_array = encode_corpus(val_examples)

print("Train tokens:", len(train_token_array))
print("Val tokens:", len(val_token_array))



Training SentencePiece tokenizer...
Tokenizer loaded.
Special tokens: {'<|system|>': 1, '<|user|>': 2, '<|assistant|>': 3, '<|end|>': 4}
Total vocab size: 32000
Train tokens: 2897411
Val tokens: 153519


In [22]:
# Save tokenized train/val corpora

train_corpus_path = os.path.join(PROJECT_ROOT, "tokenized_train_spm32k.bin")
val_corpus_path = os.path.join(PROJECT_ROOT, "tokenized_val_spm32k.bin")

with open(train_corpus_path, "wb") as f:
    f.write(train_token_array.tobytes(order="C"))
with open(val_corpus_path, "wb") as f:
    f.write(val_token_array.tobytes(order="C"))

# keep compatibility variable used by older cells
corpus_path = train_corpus_path

tokenizer_meta_path = os.path.join(PROJECT_ROOT, "tokenizer_meta_spm32k.json")
with open(tokenizer_meta_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "tokenizer_model_path": TOKENIZER_MODEL_PATH,
            "tokenizer_vocab_path": TOKENIZER_VOCAB_PATH,
            "vocab_size": VOCAB_SIZE,
            "base_vocab_size": BASE_VOCAB_SIZE,
            "special_tokens": SPECIAL_TOKENS,
            "target_vocab_size": TARGET_VOCAB_SIZE,
            "train_corpus_path": train_corpus_path,
            "val_corpus_path": val_corpus_path,
            "train_examples": len(train_examples),
            "val_examples": len(val_examples),
        },
        f,
        indent=2,
    )

print("Saved train corpus:", train_corpus_path)
print("Saved val corpus:", val_corpus_path)
print("Saved tokenizer metadata:", tokenizer_meta_path)



Saved train corpus: D:\Downloads_D\Projects\SLM\Banking_SLM_Project\tokenized_train_spm32k.bin
Saved val corpus: D:\Downloads_D\Projects\SLM\Banking_SLM_Project\tokenized_val_spm32k.bin
Saved tokenizer metadata: D:\Downloads_D\Projects\SLM\Banking_SLM_Project\tokenizer_meta_spm32k.json


In [23]:
# Training convergence stats

import math

train_tokens = int(len(train_token_array)) if 'train_token_array' in globals() else None
val_tokens = int(len(val_token_array)) if 'val_token_array' in globals() else None

block_size_now = int(globals().get('train_block_size', 64))
cpu_batch_now = int(globals().get('cpu_batch_size', 8))

tokens_per_optimizer_step = cpu_batch_now * block_size_now

train_windows = None
steps_per_epoch = None
if train_tokens is not None:
    train_windows = max(train_tokens - block_size_now - 1, 0)
    steps_per_epoch = math.ceil(train_windows / cpu_batch_now) if cpu_batch_now > 0 else None

print('Tokenizer vocab size:', VOCAB_SIZE if 'VOCAB_SIZE' in globals() else None)
print('Train examples:', len(train_examples) if 'train_examples' in globals() else None)
print('Val examples:', len(val_examples) if 'val_examples' in globals() else None)
print('Train token count:', train_tokens)
print('Val token count:', val_tokens)
print('Current block size:', block_size_now)
print('CPU batch size:', cpu_batch_now)
print('Estimated tokens per optimizer step:', tokens_per_optimizer_step)
print('Estimated train windows:', train_windows)
print('Estimated steps per epoch:', steps_per_epoch)



Tokenizer vocab size: 32000
Train examples: 27916
Val examples: 1470
Train token count: 2897411
Val token count: 153519
Current block size: 64
CPU batch size: 8
Estimated tokens per optimizer step: 512
Estimated train windows: 2897346
Estimated steps per epoch: 362169


🟠 Create Memory-Mapped Dataset Class

This ensures:
*   We never load the entire file into RAM
*   Training stays memory efficient
*   Works for large corpora






In [24]:
class MemMapDataset(Dataset):
    def __init__(self, file_path, block_size=512):
        self.data = np.memmap(file_path, dtype=np.uint32, mode='r')
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size - 1

    def __getitem__(self, idx):
        x = torch.tensor(
            self.data[idx : idx + self.block_size],
            dtype=torch.long
        )
        y = torch.tensor(
            self.data[idx + 1 : idx + self.block_size + 1],
            dtype=torch.long
        )
        return x, y

In [25]:
# Build train/val datasets from tokenized corpora

train_block_size = 64

train_dataset = MemMapDataset(train_corpus_path, block_size=train_block_size)
val_dataset = MemMapDataset(val_corpus_path, block_size=train_block_size)

# keep compatibility alias used by some older cells
dataset = train_dataset

print("Train windows:", len(train_dataset))
print("Val windows:", len(val_dataset))

x, y = train_dataset[1000]
print("X first 10:", x[:10])
print("Y first 10:", y[:10])
print("Shift correct:", torch.all(x[1:] == y[:-1]))



Train windows: 2897346
Val windows: 153454
X first 10: tensor([4764,    6,  467,    8,  155, 1887,  519,    7,  103, 2074])
Y first 10: tensor([   6,  467,    8,  155, 1887,  519,    7,  103, 2074, 2595])
Shift correct: tensor(True)


In [26]:
# intentionally left blank (cleaned)


## Build CriticalityScorer

We implement:

TRIVIAL
MEDIUM
CRITICAL


With subtypes:

policy
fraud
legal
transaction

And amount regex detection.

In [27]:
class CriticalityScorer:
    def __init__(self):
        self.policy_keywords = [
            "rbi", "policy", "circular", "regulation", "guideline"
        ]

        self.fraud_keywords = [
            "fraud", "scam", "unauthorized", "chargeback", "hack"
        ]

        self.legal_keywords = [
            "court", "legal", "compliance", "penalty"
        ]

        self.medium_keywords = [
            "loan", "interest", "fd", "emi", "account"
        ]

        # Detect large monetary amounts
        self.amount_pattern = re.compile(
            r"(₹|\$)?\s?\d{1,3}(,\d{3})*(\.\d+)?"
        )

    def score(self, text):
        text_lower = text.lower()

        # Stage 1: Keyword detection
        for word in self.policy_keywords:
            if word in text_lower:
                return "CRITICAL", "policy"

        for word in self.fraud_keywords:
            if word in text_lower:
                return "CRITICAL", "fraud"

        for word in self.legal_keywords:
            if word in text_lower:
                return "CRITICAL", "legal"

        for word in self.medium_keywords:
            if word in text_lower:
                return "MEDIUM", None

        # Stage 2: Large transaction detection
        amounts = re.findall(r"\d+(?:,\d+)?", text)
        for amt in amounts:
            value = float(amt.replace(",", ""))
            if value >= 10000:   # threshold
                return "CRITICAL", "transaction"

        return "TRIVIAL", None

In [28]:
scorer = CriticalityScorer()

tests = [
    "What are your banking hours?",
    "I want a refund for a failed transaction.",
    "There was unauthorized activity on my account.",
    "Explain your AML compliance policy.",
    "I transferred £50,000 yesterday."
]

for t in tests:
    level, subtype = scorer.score(t)
    print(f"\nText: {t}")
    print("Level:", level)
    print("Subtype:", subtype)


Text: What are your banking hours?
Level: TRIVIAL
Subtype: None

Text: I want a refund for a failed transaction.
Level: TRIVIAL
Subtype: None

Text: There was unauthorized activity on my account.
Level: CRITICAL
Subtype: fraud

Text: Explain your AML compliance policy.
Level: CRITICAL
Subtype: policy

Text: I transferred £50,000 yesterday.
Level: CRITICAL
Subtype: transaction


In [29]:
# intentionally left blank (cleaned)


## Build the Embedding Pipeline

This is where retrieval quality is determined.

Your architecture requires:

MiniLM (384-dim):

→ Linear projection (384 → 256)

→ PCA whitening (256 → 128)

→ L2 normalisation

→ Final 128-dim vector for FAISS

This solves:

Dense similarity clustering

Poor cosine spread

Signal-to-noise collapse

Long-term memory dilution

In [30]:
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

embed_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

print("MiniLM loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MiniLM loaded.


In [31]:
# Reducing the dims before PCA
import torch
import torch.nn as nn

class ProjectionLayer(nn.Module):
    def __init__(self, input_dim=384, output_dim=256):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)

        # Orthogonal initialisation
        nn.init.orthogonal_(self.linear.weight)

    def forward(self, x):
        return self.linear(x)

projection_layer = ProjectionLayer().to(device)
projection_layer.eval()

print("Projection layer ready.")

Projection layer ready.


In [32]:
# PCA Whitener
from sklearn.decomposition import PCA
import numpy as np
import random

# sample 2000 random examples
sample_texts = random.sample(all_examples, 2000)

# embed
with torch.no_grad():
    embeddings = embed_model.encode(
        sample_texts,
        convert_to_tensor=True,
        batch_size=64
    )

    projected = projection_layer(embeddings).cpu().numpy()

# Fit PCA
pca = PCA(n_components=128, whiten=True)
pca.fit(projected)

print("PCA whitener fitted.")

PCA whitener fitted.


In [33]:
import pickle

with open(f"{MEMORY_DIR}/pca_whitener.pkl", "wb") as f:
    pickle.dump(pca, f)

print("Whitener saved.")

Whitener saved.


In [34]:
# Final embed_fn for FAISS

import torch.nn.functional as F

def embed_fn(texts):
    with torch.no_grad():
        embeddings = embed_model.encode(
            texts,
            convert_to_tensor=True,
            batch_size=64
        )

        projected = projection_layer(embeddings)
        projected = projected.cpu().numpy()

        whitened = pca.transform(projected)

        # L2 normalise
        normed = whitened / np.linalg.norm(whitened, axis=1, keepdims=True)

        return normed

In [35]:
# Testing similarity seperation

vecs = embed_fn([
    "What are your branch timings?",
    "How do I report fraud?"
])

similarity = np.dot(vecs[0], vecs[1])
print("Cosine similarity:", similarity)

Cosine similarity: 0.14473125


## FAISS Partition

In [36]:
import faiss

DIM = 128

index_A = faiss.IndexFlatIP(DIM)
index_B = faiss.IndexFlatIP(DIM)
index_C = faiss.IndexFlatIP(DIM)

print("Partitions created.")

Partitions created.


In [37]:
metadata = {
    "A": [],  # policy
    "B": [],  # fraud/legal/transaction
    "C": []   # medium
}

In [38]:
from datetime import datetime
import numpy as np

class MemoryBank:
    def __init__(
        self,
        index_A=None,
        index_B=None,
        index_C=None,
        metadata=None,
        embedder=None,
        duplicate_threshold=0.92,
    ):
        # Use injected indexes/metadata when provided, otherwise fallback to globals.
        self.index_A = index_A if index_A is not None else globals().get("index_A")
        self.index_B = index_B if index_B is not None else globals().get("index_B")
        self.index_C = index_C if index_C is not None else globals().get("index_C")
        self.metadata = metadata if metadata is not None else {"A": [], "B": [], "C": []}

        # embed_fn is your sentence embedding pipeline used by FAISS.
        self.embed_fn = embedder if embedder is not None else globals().get("embed_fn")
        self.duplicate_threshold = float(duplicate_threshold)

        for key in ("A", "B", "C"):
            self.metadata.setdefault(key, [])

    def _partition_for(self, level, subtype):
        # Policy -> A, other critical items -> B, medium -> C.
        if level == "CRITICAL":
            return ("A", self.index_A) if subtype == "policy" else ("B", self.index_B)
        if level == "MEDIUM":
            return ("C", self.index_C)
        return (None, None)

    def _embed_text(self, text):
        if self.embed_fn is None:
            raise RuntimeError("embed_fn is not available. Run embedding setup cells first.")

        vector = np.asarray(self.embed_fn([text]), dtype="float32")
        if vector.ndim == 1:
            vector = vector.reshape(1, -1)

        # Normalize for cosine-similarity style retrieval.
        norms = np.linalg.norm(vector, axis=1, keepdims=True)
        vector = vector / np.clip(norms, 1e-12, None)
        return vector

    def add(self, text, level, subtype):
        partition, index = self._partition_for(level, subtype)
        if partition is None:
            return False
        if index is None:
            raise RuntimeError(f"FAISS index for partition {partition} is not initialized")

        vector = self._embed_text(text)

        # Skip near-duplicates to keep memory clean.
        if index.ntotal > 0:
            D, _ = index.search(vector, 1)
            if float(D[0][0]) >= self.duplicate_threshold:
                return False

        index.add(vector)
        self.metadata[partition].append(
            {
                "text": text,
                "subtype": subtype,
                "level": level,
                "timestamp": datetime.utcnow().isoformat(),
            }
        )
        return True

    def retrieve(self, query, k=5, partition="A", return_scores=False):
        partition = str(partition).upper()
        index = getattr(self, f"index_{partition}", None)
        meta = self.metadata.get(partition, [])

        if index is None or index.ntotal == 0:
            return []

        vector = self._embed_text(query)
        k = max(1, min(int(k), int(index.ntotal)))
        D, I = index.search(vector, k)

        results = []
        for score, idx in zip(D[0], I[0]):
            if 0 <= int(idx) < len(meta):
                item = {
                    "text": meta[int(idx)].get("text", ""),
                    "score": float(score),
                    "partition": partition,
                }
                results.append(item)

        if return_scores:
            return results
        return [r["text"] for r in results]

    def stats(self):
        a = int(self.index_A.ntotal) if self.index_A is not None else 0
        b = int(self.index_B.ntotal) if self.index_B is not None else 0
        c = int(self.index_C.ntotal) if self.index_C is not None else 0
        return {"A": a, "B": b, "C": c, "total": a + b + c}



In [39]:
memory = MemoryBank()

memory.add(
    "Our RBI policy requires KYC compliance.",
    "CRITICAL",
    "policy"
)

memory.add(
    "Customer reported fraud on credit card.",
    "CRITICAL",
    "fraud"
)

memory.add(
    "Customer asked about EMI schedule.",
    "MEDIUM",
    None
)

print("Partition A size:", memory.index_A.ntotal)
print("Partition B size:", memory.index_B.ntotal)
print("Partition C size:", memory.index_C.ntotal)

print("\nRetrieval test:")
print(memory.retrieve("Explain your KYC policy", partition="A"))

Partition A size: 1
Partition B size: 1
Partition C size: 1

Retrieval test:
['Our RBI policy requires KYC compliance.']


### Build the Audit Log

In [40]:
from datetime import datetime
import json

class AuditLog:
    def __init__(self):
        self.entries = []

    def record(self, text, level, subtype, partition, mode, decision):
        self.entries.append({
            "timestamp": str(datetime.now()),
            "preview": text[:120],
            "level": level,
            "subtype": subtype,
            "partition": partition,
            "mode": mode,
            "decision": decision
        })

    def export(self, path):
        with open(path, "w") as f:
            json.dump(self.entries, f, indent=2)

        print(f"Audit log exported to {path}")

### Approval Gate


In [41]:
class ApprovalGate:
    def __init__(self, audit_log, interactive=True, default_live_decision="n"):
        self.audit_log = audit_log
        self.interactive = bool(interactive)
        self.default_live_decision = str(default_live_decision).strip().lower() or "n"

    def _decide_yes_no(self, prompt, fallback="n"):
        fallback = str(fallback).strip().lower() or "n"
        if not self.interactive:
            return fallback == "y"

        try:
            decision = input(prompt).strip().lower()
        except EOFError:
            decision = fallback

        if decision not in {"y", "n"}:
            decision = fallback
        return decision == "y"

    def live_approve(self, text, level, subtype, partition):
        # Human approval for individual CRITICAL write.
        approved = self._decide_yes_no(
            f"\nApprove CRITICAL entry?\n{text}\n(Y/N): ",
            fallback=self.default_live_decision,
        )

        self.audit_log.record(
            text,
            level,
            subtype,
            partition,
            mode="live",
            decision="approved" if approved else "rejected",
        )
        return approved

    def batch_approve(self, entries_by_subtype, auto_approve_subtypes=None):
        approved_subtypes = set()
        auto_approve_subtypes = set(auto_approve_subtypes or [])

        for subtype, texts in entries_by_subtype.items():
            if not texts:
                continue

            if subtype in auto_approve_subtypes:
                approved = True
            else:
                approved = self._decide_yes_no(
                    f"\nSubtype: {subtype}\nCount: {len(texts)}\nSample: {texts[0][:120]}\nApprove this category? (Y/N): ",
                    fallback=self.default_live_decision,
                )

            if approved:
                approved_subtypes.add(subtype)

            for text in texts:
                self.audit_log.record(
                    text,
                    "CRITICAL",
                    subtype,
                    "A" if subtype == "policy" else "B",
                    mode="batch",
                    decision="approved" if approved else "rejected",
                )

        return approved_subtypes



In [42]:
audit_log = AuditLog()
approval_gate = ApprovalGate(audit_log, interactive=True, default_live_decision="n")


def score_and_maybe_write(text, scorer, approval_gate, memory):
    # 1) Classify the input.
    level, subtype = scorer.score(text)

    # 2) Map classification -> memory partition.
    if level == "CRITICAL":
        partition = "A" if subtype == "policy" else "B"
    elif level == "MEDIUM":
        partition = "C"
    else:
        partition = None

    # 3) Require approval only for CRITICAL writes.
    approved = True
    if level == "CRITICAL":
        approved = approval_gate.live_approve(text, level, subtype, partition)

    # 4) Write to FAISS memory only when approved and routable.
    wrote = False
    if approved and partition is not None:
        wrote = bool(memory.add(text, level, subtype))

    return {
        "text": text,
        "level": level,
        "subtype": subtype,
        "partition": partition,
        "approved": approved,
        "written": wrote,
    }



In [43]:
text = "New RBI compliance policy introduced."
result = score_and_maybe_write(text, scorer, approval_gate, memory)
print(result)



{'text': 'New RBI compliance policy introduced.', 'level': 'CRITICAL', 'subtype': 'policy', 'partition': 'A', 'approved': True, 'written': True}


In [44]:
print("Partition A size:", memory.index_A.ntotal)

Partition A size: 2


In [45]:
print(audit_log.entries[-1])

{'timestamp': '2026-03-08 12:29:31.784218', 'preview': 'New RBI compliance policy introduced.', 'level': 'CRITICAL', 'subtype': 'policy', 'partition': 'A', 'mode': 'live', 'decision': 'approved'}


## Full Dataset Memory Indexing (Pre-Training Load)

This is where your memory bank becomes institutional.

We’ll implement this cleanly but safely (chunked + batched GPU embedding).

## Two-pass logic:

### PASS 1 (CPU – fast)

Run CriticalityScorer across all_examples.
Split into:

trivial_texts

medium_texts

critical_texts (grouped by subtype)

No embeddings yet.

### PASS 2 (GPU – batched)

Embed only:

MEDIUM

CRITICAL

Then:

MEDIUM → auto add to Partition C

CRITICAL → queue for batch approval

Then:

Batch approve by subtype

Add approved entries to A or B

Finally:

Save FAISS partitions

Export audit log

In [46]:
# CPU only

from collections import defaultdict
from tqdm import tqdm

scorer = CriticalityScorer()

trivial_texts = []
medium_texts = []
critical_by_subtype = defaultdict(list)

for text in tqdm(all_examples):
    level, subtype = scorer.score(text)

    if level == "TRIVIAL":
        trivial_texts.append(text)

    elif level == "MEDIUM":
        medium_texts.append(text)

    elif level == "CRITICAL":
        critical_by_subtype[subtype].append(text)

print("Trivial:", len(trivial_texts))
print("Medium:", len(medium_texts))
print("Critical total:", sum(len(v) for v in critical_by_subtype.values()))

100%|██████████| 29386/29386 [00:00<00:00, 41159.89it/s]

Trivial: 20236
Medium: 7148
Critical total: 2002


In [47]:
# We batch embed for GPU efficiency.

def batch_embed(texts, batch_size=64):
    all_vectors = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        vecs = embed_fn(batch).astype("float32")
        all_vectors.append(vecs)
    return np.vstack(all_vectors)

In [48]:
# Embedding MEDIUM

if len(medium_texts) > 0:
    medium_vectors = batch_embed(medium_texts)

    for vec, text in zip(medium_vectors, medium_texts):
        memory.index_C.add(vec.reshape(1, -1))
        memory.metadata["C"].append({
            "text": text,
            "subtype": None,
            "timestamp": datetime.now()
        })

print("Partition C size:", memory.index_C.ntotal)

100%|██████████| 112/112 [00:51<00:00,  2.18it/s]

Partition C size: 7149


In [49]:
# Embedding CRITICAL

critical_vectors = {}

for subtype, texts in critical_by_subtype.items():
    print(f"\nEmbedding subtype: {subtype}")
    vecs = batch_embed(texts)
    critical_vectors[subtype] = vecs


Embedding subtype: transaction


100%|██████████| 3/3 [00:00<00:00,  3.03it/s]



Embedding subtype: policy


100%|██████████| 19/19 [00:08<00:00,  2.27it/s]



Embedding subtype: fraud


100%|██████████| 8/8 [00:03<00:00,  2.33it/s]



Embedding subtype: legal


100%|██████████| 4/4 [00:01<00:00,  2.28it/s]


In [50]:
approved_subtypes = approval_gate.batch_approve(critical_by_subtype)

In [51]:
# Store Approved CRITICAL

for subtype in approved_subtypes:
    texts = critical_by_subtype[subtype]
    vecs = critical_vectors[subtype]

    partition_key = "A" if subtype == "policy" else "B"
    index = getattr(memory, f"index_{partition_key}")

    for vec, text in zip(vecs, texts):
        index.add(vec.reshape(1, -1))
        memory.metadata[partition_key].append({
            "text": text,
            "subtype": subtype,
            "timestamp": datetime.now()
        })

print("Partition A:", memory.index_A.ntotal)
print("Partition B:", memory.index_B.ntotal)
print("Partition C:", memory.index_C.ntotal)

Partition A: 2
Partition B: 1
Partition C: 7149


In [52]:
import faiss
import os
import pickle


def save_memory_artifacts(memory, audit_log=None, memory_dir=MEMORY_DIR):
    # Persist FAISS indexes + metadata so retrieval works after notebook restart.
    os.makedirs(memory_dir, exist_ok=True)

    faiss.write_index(memory.index_A, os.path.join(memory_dir, "faiss_A.index"))
    faiss.write_index(memory.index_B, os.path.join(memory_dir, "faiss_B.index"))
    faiss.write_index(memory.index_C, os.path.join(memory_dir, "faiss_C.index"))

    with open(os.path.join(memory_dir, "metadata.pkl"), "wb") as f:
        pickle.dump(memory.metadata, f)

    if audit_log is not None:
        audit_log.export(os.path.join(memory_dir, "audit_log.json"))

    print("Memory artifacts saved:", memory_dir)


def load_memory_artifacts(memory_dir=MEMORY_DIR, embedder=None, duplicate_threshold=0.92):
    # Reload persisted FAISS memory state into a runtime MemoryBank object.
    path_A = os.path.join(memory_dir, "faiss_A.index")
    path_B = os.path.join(memory_dir, "faiss_B.index")
    path_C = os.path.join(memory_dir, "faiss_C.index")
    path_meta = os.path.join(memory_dir, "metadata.pkl")

    if not (os.path.exists(path_A) and os.path.exists(path_B) and os.path.exists(path_C) and os.path.exists(path_meta)):
        return None

    index_A_loaded = faiss.read_index(path_A)
    index_B_loaded = faiss.read_index(path_B)
    index_C_loaded = faiss.read_index(path_C)

    with open(path_meta, "rb") as f:
        loaded_meta = pickle.load(f)

    for key in ("A", "B", "C"):
        loaded_meta.setdefault(key, [])

    loaded_memory = MemoryBank(
        index_A=index_A_loaded,
        index_B=index_B_loaded,
        index_C=index_C_loaded,
        metadata=loaded_meta,
        embedder=embedder if embedder is not None else globals().get("embed_fn"),
        duplicate_threshold=duplicate_threshold,
    )

    stats = loaded_memory.stats()
    print(f"Loaded memory artifacts from {memory_dir} | A={stats['A']} B={stats['B']} C={stats['C']}")
    return loaded_memory


if "memory" in globals() and hasattr(memory, "index_A"):
    save_memory_artifacts(memory, audit_log=audit_log, memory_dir=MEMORY_DIR)
else:
    print("Memory object not found. Run memory setup cells before saving.")



Audit log exported to D:\Downloads_D\Projects\SLM\Banking_SLM_Project\memory_bank\audit_log.json
Memory artifacts saved: D:\Downloads_D\Projects\SLM\Banking_SLM_Project\memory_bank


## Model Configuration and Training

In [53]:
import os
import torch
from torch.utils.data import Dataset, DataLoader

class ModelConfig:
    vocab_size = VOCAB_SIZE if ("VOCAB_SIZE" in globals() and VOCAB_SIZE is not None) else 32000
    block_size = 512
    n_embd = 192
    n_head = 6
    n_layer = 6
    dropout = 0.1



We build:

Fused QKV attention

Pre-LayerNorm

Residual connections

GELU feedforward

Weight tying

Optional KV cache

In [54]:
# Causal Self-Attention (memory-efficient)

import torch.nn as nn
import torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head

        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        B, T, C = x.size()

        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=None,
            dropout_p=self.dropout.p if self.training else 0.0,
            is_causal=True,
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.dropout(self.proj(y))



In [55]:
# Transformer Block (Pre-LN)

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.n_embd)

        self.mlp = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [56]:
# Full Model with Weight Tying (low-memory training path)

from torch.utils.checkpoint import checkpoint as gradient_checkpoint

class BankingSLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.token_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Embedding(config.block_size, config.n_embd)

        self.blocks = nn.ModuleList([
            TransformerBlock(config) for _ in range(config.n_layer)
        ])

        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.head.weight = self.token_emb.weight

    def _run_blocks(self, x):
        for block in self.blocks:
            if self.training and x.requires_grad:
                x = gradient_checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        return x

    def forward(self, idx, targets=None, loss_chunk_size=8):
        B, T = idx.size()
        pos = torch.arange(0, T, device=idx.device)

        tok = self.token_emb(idx)
        pos = self.pos_emb(pos)
        x = tok + pos

        x = self._run_blocks(x)
        x = self.ln_f(x)

        if targets is None:
            logits = self.head(x)
            return logits, None

        total_loss = torch.zeros((), device=x.device)
        total_tokens = 0

        for t0 in range(0, T, loss_chunk_size):
            t1 = min(t0 + loss_chunk_size, T)
            logits_chunk = self.head(x[:, t0:t1, :])
            targets_chunk = targets[:, t0:t1]
            chunk_loss = F.cross_entropy(
                logits_chunk.reshape(-1, logits_chunk.size(-1)),
                targets_chunk.reshape(-1),
                reduction='sum'
            )
            total_loss = total_loss + chunk_loss
            total_tokens += targets_chunk.numel()

        loss = total_loss / max(total_tokens, 1)
        return None, loss

    def get_embedding(self, idx):
        with torch.no_grad():
            tok = self.token_emb(idx)
            x = tok.mean(dim=1)
            return F.normalize(x, dim=-1)



In [57]:
# Move to cuda

device = "cuda" if torch.cuda.is_available() else "cpu"

model = BankingSLM(ModelConfig()).to(device)

print("Model parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

Model parameters: 8.911872 M


In [58]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [59]:
# Optimizer + simple warmup/cosine schedule

import math
from torch.optim.lr_scheduler import LambdaLR

# Faster schedule for short runs.
learning_rate = 2e-4
max_steps = 7000
warmup_steps = 400

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=0.1,
)

def lr_lambda(step):
    if step < warmup_steps:
        return max(1e-3, (step + 1) / warmup_steps)
    progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return 0.1 + 0.9 * cosine

scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

print("LR setup:", {"lr": learning_rate, "warmup_steps": warmup_steps, "max_steps": max_steps})



LR setup: {'lr': 0.0002, 'warmup_steps': 400, 'max_steps': 7000}


In [60]:
import os
import torch

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = r"d:\Downloads_D\Projects\SLM\Banking_SLM_Project"

CHECKPOINT_PATH = os.path.join(PROJECT_ROOT, "slm_full_checkpoint.pt")
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

def save_checkpoint(model, optimizer, scheduler, step):
    special_tokens = globals().get('SPECIAL_TOKENS', {})
    base_vocab_size = globals().get('BASE_VOCAB_SIZE', model.config.vocab_size)

    checkpoint_payload = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'step': step,
        'model_config': {
            'vocab_size': model.config.vocab_size,
            'block_size': model.config.block_size,
            'n_embd': model.config.n_embd,
            'n_head': model.config.n_head,
            'n_layer': model.config.n_layer,
            'dropout': model.config.dropout,
        },
        'special_tokens': special_tokens,
        'base_vocab_size': base_vocab_size,
    }

    torch.save(checkpoint_payload, CHECKPOINT_PATH)
    print(f"Checkpoint saved at step {step}: {CHECKPOINT_PATH}")



In [61]:
# Training loop (low VRAM, periodic validation)

import gc
import math
from torch.utils.data import DataLoader

model.train()

train_block_size = 64
cpu_batch_size = 8
gpu_micro_batch_size = 1
save_every = 500
eval_every = 500

# This is optimizer-update count, not epoch count.
max_steps = 7000

# Early stopping controls (evaluated every eval_every steps).
early_stopping_patience = 6   # stop after 6 evals without improvement
early_stopping_min_delta = 1e-3

train_loader = DataLoader(
    train_dataset,
    batch_size=cpu_batch_size,
    shuffle=True,
    pin_memory=(device == 'cuda'),
    num_workers=0,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    pin_memory=(device == 'cuda'),
    num_workers=0,
)

# Estimate epochs for visibility during step-based training.
steps_per_epoch_runtime = max(1, len(train_loader))
target_epochs = max_steps / steps_per_epoch_runtime

scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda'))

print(
    f"train_block_size={train_block_size} | cpu_batch={cpu_batch_size} | "
    f"gpu_micro_batch={gpu_micro_batch_size} | steps/epoch~{steps_per_epoch_runtime} | "
    f"target_epochs~{target_epochs:.2f}"
)
print(
    f"early_stopping_patience={early_stopping_patience} | "
    f"early_stopping_min_delta={early_stopping_min_delta}"
)

@torch.no_grad()
def quick_validate(max_val_batches=200):
    model.eval()
    total_loss = 0.0
    count = 0
    for i, (vx, vy) in enumerate(val_loader):
        vx = vx.to(device, non_blocking=True)
        vy = vy.to(device, non_blocking=True)
        try:
            _, vloss = model(vx, vy, loss_chunk_size=8)
        except TypeError:
            _, vloss = model(vx, vy)
        total_loss += float(vloss.item())
        count += 1
        if i + 1 >= max_val_batches:
            break
    avg = total_loss / max(count, 1)
    ppl = math.exp(min(avg, 20))
    model.train()
    return avg, ppl

processed_tokens = 0
best_val_loss = float('inf')
best_val_ppl = float('inf')
no_improve_evals = 0
step = 0
train_iter = iter(train_loader)

while step < max_steps:
    step += 1

    try:
        x_cpu, y_cpu = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        x_cpu, y_cpu = next(train_iter)

    optimizer.zero_grad(set_to_none=True)
    batch_n = x_cpu.size(0)
    micro_steps = math.ceil(batch_n / gpu_micro_batch_size)
    step_loss = 0.0

    try:
        for m in range(micro_steps):
            s = m * gpu_micro_batch_size
            e = min((m + 1) * gpu_micro_batch_size, batch_n)

            x = x_cpu[s:e].to(device, non_blocking=True)
            y = y_cpu[s:e].to(device, non_blocking=True)

            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                try:
                    _, micro_loss = model(x, y, loss_chunk_size=8)
                except TypeError:
                    _, micro_loss = model(x, y)
                micro_loss = micro_loss / micro_steps

            step_loss += float(micro_loss.detach().cpu())
            scaler.scale(micro_loss).backward()
            del x, y, micro_loss

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

    except torch.cuda.OutOfMemoryError:
        print(f"OOM at step {step}. Clearing cache and continuing.")
        optimizer.zero_grad(set_to_none=True)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        continue

    processed_tokens += cpu_batch_size * train_block_size

    if step % 50 == 0:
        lr_now = optimizer.param_groups[0]['lr']
        approx_epoch = step / steps_per_epoch_runtime
        if torch.cuda.is_available():
            alloc = torch.cuda.memory_allocated() / 1024**2
            print(f"Step {step} | Epoch~{approx_epoch:.2f} | TrainLoss {step_loss:.4f} | LR {lr_now:.2e} | tokens {processed_tokens:,} | alloc {alloc:.0f} MB")
        else:
            print(f"Step {step} | Epoch~{approx_epoch:.2f} | TrainLoss {step_loss:.4f} | LR {lr_now:.2e} | tokens {processed_tokens:,}")

    if step % eval_every == 0:
        val_loss, val_ppl = quick_validate(max_val_batches=200)
        print(f"Step {step} | ValLoss {val_loss:.4f} | ValPPL {val_ppl:.2f}")

        improved = val_loss < (best_val_loss - early_stopping_min_delta)
        if improved:
            best_val_loss = val_loss
            best_val_ppl = val_ppl
            no_improve_evals = 0
            save_checkpoint(model, optimizer, scheduler, step)
            print(f"New best checkpoint at step {step} (ValLoss={val_loss:.4f}, ValPPL={val_ppl:.2f})")
        else:
            no_improve_evals += 1
            print(f"No val improvement for {no_improve_evals}/{early_stopping_patience} eval checks")

            if no_improve_evals >= early_stopping_patience:
                print(f"Early stopping triggered at step {step}")
                break

    if step % save_every == 0:
        save_checkpoint(model, optimizer, scheduler, step)

save_checkpoint(model, optimizer, scheduler, step)
print(f"Training done at step {step}. Best ValLoss: {best_val_loss:.4f} | Best ValPPL: {best_val_ppl:.2f}")



train_block_size=64 | cpu_batch=8 | gpu_micro_batch=1 | steps/epoch~362168 | target_epochs~0.02
early_stopping_patience=6 | early_stopping_min_delta=0.001
Step 50 | Epoch~0.00 | TrainLoss 109.6776 | LR 2.55e-05 | tokens 25,600 | alloc 244 MB
Step 100 | Epoch~0.00 | TrainLoss 48.1002 | LR 5.05e-05 | tokens 51,200 | alloc 244 MB
Step 150 | Epoch~0.00 | TrainLoss 31.4063 | LR 7.55e-05 | tokens 76,800 | alloc 244 MB
Step 200 | Epoch~0.00 | TrainLoss 25.7562 | LR 1.00e-04 | tokens 102,400 | alloc 244 MB
Step 250 | Epoch~0.00 | TrainLoss 24.1261 | LR 1.25e-04 | tokens 128,000 | alloc 244 MB
Step 300 | Epoch~0.00 | TrainLoss 22.9240 | LR 1.50e-04 | tokens 153,600 | alloc 244 MB
Step 350 | Epoch~0.00 | TrainLoss 19.8004 | LR 1.76e-04 | tokens 179,200 | alloc 244 MB
Step 400 | Epoch~0.00 | TrainLoss 17.8744 | LR 2.00e-04 | tokens 204,800 | alloc 244 MB
Step 450 | Epoch~0.00 | TrainLoss 17.1408 | LR 2.00e-04 | tokens 230,400 | alloc 244 MB
Step 500 | Epoch~0.00 | TrainLoss 15.5231 | LR 2.00e-04 

In [62]:
print("Vocab size:", model.config.vocab_size)
print("Embedding shape:", model.token_emb.weight.shape)

Vocab size: 32000
Embedding shape: torch.Size([32000, 192])


In [63]:
# Load model checkpoint for quick demo

ckpt = torch.load(CHECKPOINT_PATH, map_location=torch.device("cpu"))
model = BankingSLM(ModelConfig()).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print("Loaded checkpoint:", CHECKPOINT_PATH)



C:\Users\HP\AppData\Local\Temp\ipykernel_11796\663677058.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CHECKPOINT_PATH, map_location=torch.device("cp

Loaded checkpoint: D:\Downloads_D\Projects\SLM\Banking_SLM_Project\slm_full_checkpoint.pt


In [64]:
# Memory-grounded generation + quick sanity checks

import re

SPECIAL_TAG_RE = re.compile(r"<\|[^|]+\|>")


def _strip_special_tags(text):
    # Remove template tags before semantic retrieval query.
    return SPECIAL_TAG_RE.sub(" ", text).strip()


def ensure_memory_runtime_loaded():
    # Reuse existing in-memory bank if already loaded and non-empty.
    if "memory" in globals() and hasattr(memory, "stats"):
        stats = memory.stats()
        if stats.get("total", 0) > 0:
            return True

    # Otherwise, try loading saved FAISS artifacts.
    if "load_memory_artifacts" in globals():
        loaded = load_memory_artifacts(
            memory_dir=MEMORY_DIR,
            embedder=globals().get("embed_fn"),
        )
        if loaded is not None:
            globals()["memory"] = loaded
            return True

    return False


def retrieve_memory_context(query, k_per_partition=2, max_items=6):
    # Fetch top memories from A/B/C, merge, sort by similarity, then dedupe.
    if not ensure_memory_runtime_loaded():
        return []

    retrieved = []
    for partition in ["A", "B", "C"]:
        try:
            part_items = memory.retrieve(
                query,
                k=k_per_partition,
                partition=partition,
                return_scores=True,
            )
        except TypeError:
            # Backward compatibility with older retrieve() signatures.
            texts = memory.retrieve(query, k=k_per_partition, partition=partition)
            part_items = [{"text": t, "score": 0.0, "partition": partition} for t in texts]
        except Exception:
            part_items = []

        retrieved.extend(part_items)

    retrieved.sort(key=lambda x: float(x.get("score", 0.0)), reverse=True)

    deduped = []
    seen = set()
    for item in retrieved:
        text = str(item.get("text", "")).strip()
        key = text.lower()
        if text and key not in seen:
            seen.add(key)
            deduped.append(text)

    return deduped[:max_items]


@torch.no_grad()
def generate(
    model,
    prompt,
    max_new_tokens=100,
    temperature=0.7,
    top_k=40,
    use_memory=True,
    k_per_partition=2,
):
    model.eval()

    # Wrap plain input in your chat template.
    if "<|user|>" in prompt and "<|assistant|>" in prompt:
        base_prompt = prompt.strip()
    else:
        base_prompt = f"<|user|> {prompt.strip()} <|assistant|>"

    if use_memory:
        query = _strip_special_tags(base_prompt)
        snippets = retrieve_memory_context(query, k_per_partition=k_per_partition)

        if snippets:
            memory_text = "\n".join([f"- {s}" for s in snippets])
            memory_prefix = (
                "<|system|> Use retrieved memory if relevant. "
                "If memory is missing, answer conservatively.\n"
                f"{memory_text} <|end|> "
            )
            prompt_for_model = memory_prefix + base_prompt
        else:
            prompt_for_model = base_prompt
    else:
        prompt_for_model = base_prompt

    input_ids = torch.tensor(tokenize(prompt_for_model), dtype=torch.long).unsqueeze(0).to(device)

    for _ in range(max_new_tokens):
        input_ids = input_ids[:, -model.config.block_size:]

        if device == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits, _ = model(input_ids)
        else:
            logits, _ = model(input_ids)

        logits = logits[:, -1, :]

        if temperature is not None and temperature > 0:
            logits = logits / temperature

        if top_k is not None and top_k > 0:
            k = min(int(top_k), logits.size(-1))
            values, indices = torch.topk(logits, k, dim=-1)
            probs = torch.softmax(values, dim=-1)
            sampled_k_idx = torch.multinomial(probs, num_samples=1)
            next_token = indices.gather(-1, sampled_k_idx)
        else:
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat([input_ids, next_token], dim=1)

    return detokenize(input_ids[0].tolist())


print(generate(model, "What is RBI KYC policy?", max_new_tokens=60, temperature=0.7, top_k=40, use_memory=True))
print(generate(model, "KYC stands for", max_new_tokens=40, temperature=0.7, top_k=40, use_memory=True))



Use retrieved memory if relevant. If memory is missing, answer conservatively. - Our RBI policy requires KYC compliance. - New RBI compliance policy introduced. -  $MCIG - MCig up 9% on CBD distribution deals https://t.co/WV5yrJFdyD  Sentiment: neutral.  -  Analyze the political scenario of a given country India  India is a federal parliamentary democratic republic, and as such its political scenario is influenced by different branches of government, political parties, institutions, and the public. The President of India serves as the nominal head of state, while the Prime Minister is the head of government and exercises most executive power, leading the central government and making decisions in consultation with their Cabinet. Currently, the President of India is Ram Nath Kovind, while the Prime Minister is Narendra Modi. In India's parliamentary system, the ruling government is formed by the party or coalition that has the majority of seats in the Lok Sabha, the lower house of Parli

In [65]:
# Manual validation perplexity

import math
from torch.utils.data import DataLoader

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=(device == 'cuda'),
)

@torch.no_grad()
def evaluate_perplexity(max_batches=300):
    model.eval()
    total_loss = 0.0
    count = 0

    for i, (x, y) in enumerate(val_loader):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        try:
            _, loss = model(x, y, loss_chunk_size=8)
        except TypeError:
            _, loss = model(x, y)

        total_loss += float(loss.item())
        count += 1

        if i + 1 >= max_batches:
            break

    avg_loss = total_loss / max(count, 1)
    ppl = math.exp(min(avg_loss, 20))
    return avg_loss, ppl, count

val_loss, val_ppl, val_batches = evaluate_perplexity(max_batches=300)
print("Validation batches:", val_batches)
print("Validation avg loss:", val_loss)
print("Validation perplexity:", val_ppl)



Validation batches: 300
Validation avg loss: 5.959907725652059
Validation perplexity: 387.57435941642706


## Chat Test Section

Use this interactive loop to test the SLM with memory-grounded generation.
Type `exit` to stop.

In [66]:
# Interactive chat loop for quick SLM testing

if "model" not in globals():
    raise RuntimeError("Model not found. Run the checkpoint load cell first.")
if "generate" not in globals():
    raise RuntimeError("generate() not found. Run the memory-grounded generation cell first.")

print("SLM chat ready. Type 'exit' to stop.\n")

while True:
    user_text = input("You: ").strip()

    if user_text.lower() in {"exit", "quit"}:
        print("Chat ended.")
        break

    if not user_text:
        continue

    answer = generate(
        model,
        user_text,
        max_new_tokens=120,
        temperature=0.7,
        top_k=40,
        use_memory=True,
        k_per_partition=2,
    )

    # Keep only assistant tail when template tokens are present.
    if "<|assistant|>" in answer:
        answer = answer.split("<|assistant|>")[-1].strip()

    print(f"SLM: {answer}\n")



SLM chat ready. Type 'exit' to stop.

SLM: is a vast region that comprises of many countries, and as such, the relationships between these countries are complex and multi-faceted. Some of the countries in Asia have good diplomatic, economic and cultural relations, while some have a history of conflicts and disputes. For instance, countries in South East Asia such as Thailand, Vietnam, Indonesia, Malaysia, the Philippines, Singapore, Brunei, Laos, Myanmar, and Cambodia have a regional intergovernmental organization called the Association of Southeast Asian Nations (ASEAN), which promotes economic, political, and security cooperation among its members. Similarly, Japan, South Korea, and China, three of the dominant economies in East Asia, are highly interdependent, with a large amount of trade and investment flowing between them. They also share historical and cultural ties, with a common Confucian heritage. On the other hand, there are long-standing tensions between North and South Kore